# C3 — Colectare comentarii YouTube
În acest notebook colectăm un eșantion  de comentarii publice de pe YouTube.
Scopul nu este să obținem corpusul final mare, ci să înțelegem fluxul:
sursă → API → comentarii brute → fișier JSONL.
La final, fiecare student salvează propriul fișier în `data/raw/`.

## 1. Ce trebuie să avem pregătit
Avem nevoie de:
- fișier `.env` în root-ul proiectului
- cheia `YOUTUBE_API_KEY`
- un handle de canal YouTube
Exemplu în `.env`:
```text
YOUTUBE_API_KEY=cheia_ta_aici

In [4]:

from pathlib import Path
import os
import json
import requests
from datetime import datetime
from dotenv import load_dotenv

## 2. Încărcăm cheia API
Notebook-ul caută fișierul `.env` în root-ul proiectului.
Dacă cheia nu este găsită, colectarea nu poate porni.

In [5]:
ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")
API_KEY = os.getenv("YOUTUBE_API_KEY")
BASE_URL = "https://www.googleapis.com/youtube/v3"
print("Root proiect:", ROOT)
print("Cheie găsită:", API_KEY is not None)

Root proiect: c:\Users\ASUS\Desktop\ADC 2\INGINERIE AI\echochamber-project-team-2
Cheie găsită: True


## 3. Alegem canalul și numărul de videoclipuri
Fiecare student schimbă `student_id` și `handle`.
Pentru exercițiu folosim puține videoclipuri, ca să nu consumăm inutil cota API.

In [6]:
student_id = "student_03"
handle = "@valeriuciolannicolae-s8h"
max_videos = 10
max_comments_per_video = 10
output_file = ROOT / "data" / "raw" / f"{student_id}_youtube_raw.jsonl"
print(output_file)

c:\Users\ASUS\Desktop\ADC 2\INGINERIE AI\echochamber-project-team-2\data\raw\student_03_youtube_raw.jsonl


## 4. Găsim canalul YouTube

YouTube lucrează intern cu `channel_id`, nu direct cu numele canalului.
De aceea, primul pas este să transformăm handle-ul în `channel_id`.

In [7]:
channel_response = requests.get(
    f"{BASE_URL}/channels",
    params={
        "part": "id",
        "forHandle": handle,
        "key": API_KEY
    }
)
channel_data = channel_response.json()
channel_data

{'kind': 'youtube#channelListResponse',
 'etag': 'hsBGRkSKWfzNyscElNHk-pfADno',
 'pageInfo': {'totalResults': 1, 'resultsPerPage': 5},
 'items': [{'kind': 'youtube#channel',
   'etag': '8g_7KL1SaXoiRXKbqEEwPsOJFfw',
   'id': 'UCByIhdIhp5b37tTjJQxln1w'}]}

In [8]:
channel_id = channel_data["items"][0]["id"]
channel_id

'UCByIhdIhp5b37tTjJQxln1w'

## 5. Luăm cele mai recente videoclipuri
Acum cerem ultimele videoclipuri publicate de canal.
Pentru curs folosim doar câteva videoclipuri.

In [9]:
videos_response = requests.get(
    f"{BASE_URL}/search",
    params={
        "part": "snippet",
        "channelId": channel_id,
        "type": "video",
        "order": "date",
        "maxResults": max_videos,
        "key": API_KEY
    }
)
videos_data = videos_response.json()
videos_data["items"][0]

{'kind': 'youtube#searchResult',
 'etag': 'zchHnDQSn6zdTz2LRRGkfYOBKxE',
 'id': {'kind': 'youtube#video', 'videoId': 'ypgxYxyoNcg'},
 'snippet': {'publishedAt': '2026-05-08T16:30:06Z',
  'channelId': 'UCByIhdIhp5b37tTjJQxln1w',
  'title': 'Budeanu, Ghiță, Ponta, off-shore-uri, miliarde în contracte cu statul, săgeți din parlament și Roman',
  'description': 'Despre legăturile dintre case de avocatură, foști gineri de lideri comuniști, servicii secrete, presă, televiziuni, PSD și un pic și ...',
  'thumbnails': {'default': {'url': 'https://i.ytimg.com/vi/ypgxYxyoNcg/default.jpg',
    'width': 120,
    'height': 90},
   'medium': {'url': 'https://i.ytimg.com/vi/ypgxYxyoNcg/mqdefault.jpg',
    'width': 320,
    'height': 180},
   'high': {'url': 'https://i.ytimg.com/vi/ypgxYxyoNcg/hqdefault.jpg',
    'width': 480,
    'height': 360}},
  'channelTitle': 'Valeriu Nicolae',
  'liveBroadcastContent': 'none',
  'publishTime': '2026-05-08T16:30:06Z'}}

In [10]:
videos = []
for item in videos_data["items"]:
    videos.append({
        "video_id": item["id"]["videoId"],
        "video_title": item["snippet"]["title"],
        "video_date": item["snippet"]["publishedAt"][:10]
    })
videos

[{'video_id': 'ypgxYxyoNcg',
  'video_title': 'Budeanu, Ghiță, Ponta, off-shore-uri, miliarde în contracte cu statul, săgeți din parlament și Roman',
  'video_date': '2026-05-08'},
 {'video_id': 'oEyP2p1mfcE',
  'video_title': 'Călin Georgescu și dublul standard: Reguli pentru alții, nu pentru el',
  'video_date': '2026-05-08'},
 {'video_id': 'JOx27O7fQ_E',
  'video_title': 'Mâine întregul episod. Azi doar un mic teaser. E pistol cu apă față de ce vine mâine. Promit.',
  'video_date': '2026-05-07'},
 {'video_id': 'LSRRUQ7m9xM',
  'video_title': 'Călin Georgescu: „Cheia se află la Kogălniceanu”',
  'video_date': '2026-05-07'},
 {'video_id': 'ENUKTEjGNsE',
  'video_title': 'Momentul în care Turcescu a fost „trezit în conștiință” de Georgescu',
  'video_date': '2026-05-06'},
 {'video_id': '78p1fI_OCbI',
  'video_title': 'Călin Georgescu și „vestile bune” despre lucruri rele',
  'video_date': '2026-05-05'},
 {'video_id': 'DJyiSIo7B74',
  'video_title': 'Inteligența Artificială vs  Călin Ge

## 6. Colectăm comentariile
Pentru fiecare videoclip luăm comentariile publice ordonate după relevanță.
În acest exercițiu nu folosim paginare, deci luăm maximum 100 comentarii per videoclip.

In [11]:
comments = []
for video in videos:
    print("Colectez:", video["video_title"][:80])
    comments_response = requests.get(
        f"{BASE_URL}/commentThreads",
        params={
            "part": "snippet",
            "videoId": video["video_id"],
            "maxResults": max_comments_per_video,
            "textFormat": "plainText",
            "order": "relevance",
            "key": API_KEY
        }
    )
    comments_data = comments_response.json()
    for comment_item in comments_data.get("items", []):
        snippet = comment_item["snippet"]["topLevelComment"]["snippet"]
        record = {
            "id": f"yt_{video['video_id']}_{comment_item['id']}",
            "source_platform": "youtube",
            "source_channel": handle,
            "text_raw": snippet["textDisplay"],
            "video_id": video["video_id"],
            "video_title": video["video_title"],
            "video_date": video["video_date"],
            "comment_date": snippet["publishedAt"][:10],
            "likes": snippet["likeCount"],
            "collected_at": datetime.utcnow().strftime("%Y-%m-%d")
        }
        comments.append(record)
len(comments)

Colectez: Budeanu, Ghiță, Ponta, off-shore-uri, miliarde în contracte cu statul, săgeți di


C:\Users\ASUS\AppData\Local\Temp\ipykernel_18280\3206550858.py:28: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "collected_at": datetime.utcnow().strftime("%Y-%m-%d")


Colectez: Călin Georgescu și dublul standard: Reguli pentru alții, nu pentru el
Colectez: Mâine întregul episod. Azi doar un mic teaser. E pistol cu apă față de ce vine m
Colectez: Călin Georgescu: „Cheia se află la Kogălniceanu”
Colectez: Momentul în care Turcescu a fost „trezit în conștiință” de Georgescu
Colectez: Călin Georgescu și „vestile bune” despre lucruri rele
Colectez: Inteligența Artificială vs  Călin Georgescu:  Ne întoarcem la origini?
Colectez: Moțiune, ticăloșia PSD, de ce nu e Bolojan sfânt, lașitatea lui Nicușor și de ce
Colectez: Cine sunt antihriștii cu care se luptă Călin Georgescu 👹
Colectez: Călin Georgescu și profeția de pe 6 Decembrie: Politică sau misticism?


100

# Explorare si curatare

## 7. Inspectăm primele comentarii
Înainte să salvăm fișierul, verificăm dacă datele arată cum trebuie.

In [12]:
comments[:3]

[{'id': 'yt_ypgxYxyoNcg_Ugzj4_KpV4fxwAHy0Ex4AaABAg',
  'source_platform': 'youtube',
  'source_channel': '@valeriuciolannicolae-s8h',
  'text_raw': 'Asta a fost o documentare tare grea. Dacă vreți să ne ajutați puteți să ne fiți moguli - ASOCIAȚIA CASA BUNĂ  - Iban RO63BTRLRONCRT0566398304 sau cu 3.5% - https://redirectioneaza.ro/asociatia-casa-buna/. Mulțumim!',
  'video_id': 'ypgxYxyoNcg',
  'video_title': 'Budeanu, Ghiță, Ponta, off-shore-uri, miliarde în contracte cu statul, săgeți din parlament și Roman',
  'video_date': '2026-05-08',
  'comment_date': '2026-05-08',
  'likes': 76,
  'collected_at': '2026-05-09'},
 {'id': 'yt_ypgxYxyoNcg_UgwxGd6GhCqxhXbpXE94AaABAg',
  'source_platform': 'youtube',
  'source_channel': '@valeriuciolannicolae-s8h',
  'text_raw': 'Sa trăiască domnul Valeriu Nicolae 100 de ani!',
  'video_id': 'ypgxYxyoNcg',
  'video_title': 'Budeanu, Ghiță, Ponta, off-shore-uri, miliarde în contracte cu statul, săgeți din parlament și Roman',
  'video_date': '2026-05-0

In [13]:
comments[0].keys()

dict_keys(['id', 'source_platform', 'source_channel', 'text_raw', 'video_id', 'video_title', 'video_date', 'comment_date', 'likes', 'collected_at'])

## 8. Curățare minimă a textului
Acum pornim de la `text_raw` și construim o variantă curățată în câmpul `text`.
Nu schimbăm sensul comentariului. Eliminăm doar zgomot simplu: linkuri, spații inutile, texte prea scurte și duplicate.

In [14]:
import re

def clean_text(text):
    text = re.sub(r"http\S+", "", text)      # elimină linkuri
    text = re.sub(r"\s+", " ", text)         # normalizează spațiile
    return text.strip()

## 9. Aplicăm curățarea
Pentru fiecare comentariu păstrăm textul original în `text_raw` și adăugăm textul curățat în `text`.

In [15]:
for comment in comments:
    comment["text"] = clean_text(comment["text_raw"])

comments[0]

{'id': 'yt_ypgxYxyoNcg_Ugzj4_KpV4fxwAHy0Ex4AaABAg',
 'source_platform': 'youtube',
 'source_channel': '@valeriuciolannicolae-s8h',
 'text_raw': 'Asta a fost o documentare tare grea. Dacă vreți să ne ajutați puteți să ne fiți moguli - ASOCIAȚIA CASA BUNĂ  - Iban RO63BTRLRONCRT0566398304 sau cu 3.5% - https://redirectioneaza.ro/asociatia-casa-buna/. Mulțumim!',
 'video_id': 'ypgxYxyoNcg',
 'video_title': 'Budeanu, Ghiță, Ponta, off-shore-uri, miliarde în contracte cu statul, săgeți din parlament și Roman',
 'video_date': '2026-05-08',
 'comment_date': '2026-05-08',
 'likes': 76,
 'collected_at': '2026-05-09',
 'text': 'Asta a fost o documentare tare grea. Dacă vreți să ne ajutați puteți să ne fiți moguli - ASOCIAȚIA CASA BUNĂ - Iban RO63BTRLRONCRT0566398304 sau cu 3.5% - Mulțumim!'}

## 10. Filtrăm comentariile prea scurte
Pentru exercițiu păstrăm doar comentariile care au cel puțin 60 de caractere.
Comentariile foarte scurte sunt greu de interpretat în analiza discursivă.

In [16]:
MIN_CHARS = 60

comments_clean = [
    comment for comment in comments
    if len(comment["text"]) >= MIN_CHARS
]

print("Comentarii brute:", len(comments))
print("Comentarii după filtrarea lungimii:", len(comments_clean))

Comentarii brute: 100
Comentarii după filtrarea lungimii: 62


## 11. Filtrăm textele cu prea puține litere
Comentariile formate mai ales din emoji, simboluri sau caractere izolate produc zgomot.
Păstrăm comentariile în care cel puțin 50% dintre caractere sunt litere.

In [17]:
MIN_ALPHA = 0.5

def alpha_ratio(text):
    if len(text) == 0:
        return 0
    letters = sum(char.isalpha() for char in text)
    return letters / len(text)

comments_clean = [
    comment for comment in comments_clean
    if alpha_ratio(comment["text"]) >= MIN_ALPHA
]

print("Comentarii după filtrarea literelor:", len(comments_clean))

Comentarii după filtrarea literelor: 62


## 12. Eliminăm duplicatele
Dacă același text apare de mai multe ori, îl păstrăm o singură dată.

In [18]:
seen_texts = set()
unique_comments = []

for comment in comments_clean:
    text = comment["text"].lower()
    if text not in seen_texts:
        unique_comments.append(comment)
        seen_texts.add(text)

comments_clean = unique_comments

print("Comentarii finale după deduplicare:", len(comments_clean))

Comentarii finale după deduplicare: 62


## 14. Salvăm fișierul curățat
Salvăm rezultatul în `data/cleaned/`.

In [16]:
clean_output_file = ROOT / "data" / "cleaned" / f"{student_id}_youtube_clean.jsonl"
clean_output_file.parent.mkdir(parents=True, exist_ok=True)

with clean_output_file.open("w", encoding="utf-8") as f:
    for comment in comments_clean:
        f.write(json.dumps(comment, ensure_ascii=False) + "\n")

print("Comentarii curate salvate:", len(comments_clean))
print("Fișier:", clean_output_file)

Comentarii curate salvate: 65
Fișier: c:\Users\ASUS\Desktop\ADC 2\INGINERIE AI\echochamber-project-team-2\data\cleaned\student_03_youtube_clean.jsonl


# Functia de curatare

In [25]:
import re

# remove emojis
def remove_emojis(text):
    return re.sub(
        "["
        "\U0001F1E0-\U0001F1FF"  # flags
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F680-\U0001F6FF"  # transport & map
        "\U0001F700-\U0001F77F"
        "\U0001F780-\U0001F7FF"
        "\U0001F800-\U0001F8FF"
        "\U0001F900-\U0001F9FF"  # supplemental symbols
        "\U0001FA00-\U0001FAFF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+",
        "",
        text,
        flags=re.UNICODE
    )

#cuvinte lipite

def fix_spacing_typos(text):
    replacements = {
        r"\bpânăla\b": "până la",
        r"\bpânala\b": "până la",
        r"\bpânăîn\b": "până în",
        r"\bpânăpe\b": "până pe",
        r"\bdela\b": "de la",
        r"\dintro\b": "dintr-o",
        r"\dintrun\b": "dintr-un",
        r"\întrun\b": "într-un",
        r"\întro\b": "într-o",
    }

    for pattern, replacement in replacements.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)

    return text

def clean_comments(comments, min_chars=60, min_alpha=0.5):
    cleaned = []
    seen_texts = set()
    
    for comment in comments:
        # 1. Curățare text
        text = comment["text_raw"]
        text = re.sub(r"http\S+", "", text)
        text = remove_emojis(text)
        text = fix_spacing_typos(text)
        text = re.sub(r"\s+", " ", text).strip()
        
        # 2. Filtru lungime
        if len(text) < min_chars:
            continue
        
        # 3. Filtru proporție litere
        letters = sum(char.isalpha() for char in text)
        alpha_ratio = letters / len(text) if len(text) > 0 else 0
        
        if alpha_ratio < min_alpha:
            continue
        
        # 4. Filtru duplicate
        text_key = text.lower()
        if text_key in seen_texts:
            continue
        
        seen_texts.add(text_key)
        
        # 5. Păstrăm comentariul și adăugăm textul curățat
        new_comment = comment.copy()
        new_comment["text"] = text
        new_comment["lang"] = "ro"
        cleaned.append(new_comment)
    
    return cleaned


In [26]:
comments_clean = clean_comments(
    comments,
    min_chars=60,
    min_alpha=0.5
)

print("Comentarii brute:", len(comments))
print("Comentarii curate:", len(comments_clean))

Comentarii brute: 100
Comentarii curate: 62


In [27]:
for comment in comments_clean[:3]:
    print("RAW:", comment["text_raw"])
    print("CLEAN:", comment["text"])
    print("---")

RAW: Asta a fost o documentare tare grea. Dacă vreți să ne ajutați puteți să ne fiți moguli - ASOCIAȚIA CASA BUNĂ  - Iban RO63BTRLRONCRT0566398304 sau cu 3.5% - https://redirectioneaza.ro/asociatia-casa-buna/. Mulțumim!
CLEAN: Asta a fost o documentare tare grea. Dacă vreți să ne ajutați puteți să ne fiți moguli - ASOCIAȚIA CASA BUNĂ - Iban RO63BTRLRONCRT0566398304 sau cu 3.5% - Mulțumim!
---
RAW: În fiecare zi aflăm mizerii despre gașca de hoți! Și totuși ei au un tupeu inimaginabil!
CLEAN: În fiecare zi aflăm mizerii despre gașca de hoți! Și totuși ei au un tupeu inimaginabil!
---
RAW: Impresionantă documentarea ca noi să putem vedea adevărul despre cei ce ne conduc 😢😢😢
CLEAN: Impresionantă documentarea ca noi să putem vedea adevărul despre cei ce ne conduc
---


In [28]:
for comment in comments_clean[:6]:
    print("RAW:", comment["text_raw"])
    print("CLEAN:", comment["text"])
    print("---")

RAW: Asta a fost o documentare tare grea. Dacă vreți să ne ajutați puteți să ne fiți moguli - ASOCIAȚIA CASA BUNĂ  - Iban RO63BTRLRONCRT0566398304 sau cu 3.5% - https://redirectioneaza.ro/asociatia-casa-buna/. Mulțumim!
CLEAN: Asta a fost o documentare tare grea. Dacă vreți să ne ajutați puteți să ne fiți moguli - ASOCIAȚIA CASA BUNĂ - Iban RO63BTRLRONCRT0566398304 sau cu 3.5% - Mulțumim!
---
RAW: În fiecare zi aflăm mizerii despre gașca de hoți! Și totuși ei au un tupeu inimaginabil!
CLEAN: În fiecare zi aflăm mizerii despre gașca de hoți! Și totuși ei au un tupeu inimaginabil!
---
RAW: Impresionantă documentarea ca noi să putem vedea adevărul despre cei ce ne conduc 😢😢😢
CLEAN: Impresionantă documentarea ca noi să putem vedea adevărul despre cei ce ne conduc
---
RAW: M-a luat cu ameteala, greu de inghitit realitatea prezentata de dumneavoastra. Sanse de iesit la lumina  nu vad in timpul vietii mele, probleme este ca nu vad lumina nici pentru nepotii mei.
CLEAN: M-a luat cu ameteala, g

In [22]:
clean_output_file = ROOT / "data" / "cleaned" / f"{student_id}_youtube_clean.jsonl"
clean_output_file.parent.mkdir(parents=True, exist_ok=True)

with clean_output_file.open("w", encoding="utf-8") as f:
    for comment in comments_clean:
        f.write(json.dumps(comment, ensure_ascii=False) + "\n")

print("Fișier salvat:", clean_output_file)
print("Comentarii salvate:", len(comments_clean))

Fișier salvat: c:\Users\ASUS\Desktop\ADC 2\INGINERIE AI\echochamber-project-team-2\data\cleaned\student_03_youtube_clean.jsonl
Comentarii salvate: 62


15. Ce am obținut
Am produs două fișiere:
- `data/raw/student_XX_youtube_raw.jsonl` — comentarii brute
- `data/cleaned/student_XX_youtube_clean.jsonl` — comentarii curățate
Fișierul curățat va putea fi unit cu fișierele celorlalți membri ai echipei.